# Notebook 07: Model Monitoring & Retraining

**Purpose**: Establish model performance monitoring, drift detection, and automated retraining pipeline.

**Objectives**:
- Monitor model performance over time
- Detect concept drift and data drift
- Implement retraining triggers and pipelines
- A/B test new model versions

**Dependencies**: Notebook 04 (trained models), MLflow

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Project imports
from src.etl.extractors import extract_all
from src.etl.transformers import transform_all
from src.features.aggregations import create_fraud_detection_dataset
from src.config.viz_theme import (
    PROJECT_THEME,
    REGION_COLORS,
    get_highlight_colors,
    get_top_n_highlight_colors,
    apply_project_theme,
    get_label,
)

print("Imports loaded successfully")

## 2. Load Data & Models

In [ ]:
# Extract and transform data
raw_data = extract_all()
clean_data = transform_all(raw_data)

orders = clean_data['orders']
drivers = clean_data['drivers']
customers = clean_data['customers']
products = clean_data['products']
missing_items = clean_data['missing_items']

print(f"Orders loaded: {len(orders):,}")
print(f"Date range: {orders['order_date'].min()} to {orders['order_date'].max()}")

In [ ]:
# Create ML dataset
ml_data = create_fraud_detection_dataset(clean_data)
print(f"ML dataset shape: {ml_data.shape}")
print(f"Features available: {ml_data.columns.tolist()}")

In [ ]:
# Define features for modeling
FEATURE_COLUMNS = [
    'order_amount', 'items_delivered', 'items_missing', 'total_items', 'missing_rate',
    'driver_age', 'trips', 'driver_missing_rate', 'driver_total_orders',
    'customer_age', 'customer_missing_rate', 'customer_total_orders'
]

# Get available features
available_features = [f for f in FEATURE_COLUMNS if f in ml_data.columns]
print(f"Using {len(available_features)} features: {available_features}")

## 3. Performance Monitoring

Track model predictions and anomaly rates over time to detect degradation.

In [ ]:
# Prepare features
X = ml_data[available_features].copy()
X = X.fillna(X.median())

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train baseline Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)
iso_forest.fit(X_scaled)

# Get predictions and scores
ml_data['anomaly_pred'] = iso_forest.predict(X_scaled)
ml_data['anomaly_score'] = iso_forest.decision_function(X_scaled)

anomaly_rate = (ml_data['anomaly_pred'] == -1).mean() * 100
print(f"Baseline anomaly rate: {anomaly_rate:.2f}%")

In [ ]:
# Simulate monthly monitoring by splitting data into time periods
ml_data['order_month'] = pd.to_datetime(ml_data['order_date']).dt.to_period('M')

monthly_metrics = ml_data.groupby('order_month').agg({
    'anomaly_pred': lambda x: (x == -1).mean() * 100,
    'anomaly_score': ['mean', 'std'],
    'missing_rate': 'mean',
    'order_id': 'count'
}).reset_index()

monthly_metrics.columns = ['month', 'anomaly_rate', 'avg_score', 'score_std', 'avg_missing_rate', 'order_count']
monthly_metrics['month'] = monthly_metrics['month'].astype(str)

print("Monthly performance metrics:")
monthly_metrics

In [ ]:
# Anomaly Rate Over Time
fig = px.line(
    monthly_metrics,
    x='month',
    y='anomaly_rate',
    title='Model Anomaly Rate Over Time',
    labels={'month': 'Month', 'anomaly_rate': 'Anomaly Rate (%)'},
    markers=True
)

# Add threshold line
avg_rate = monthly_metrics['anomaly_rate'].mean()
fig.add_hline(
    y=avg_rate,
    line_dash='dash',
    line_color=PROJECT_THEME['neutral_gray'],
    annotation_text=f'Average: {avg_rate:.1f}%'
)

# Add alert thresholds
fig.add_hline(
    y=avg_rate * 1.2,
    line_dash='dot',
    line_color=PROJECT_THEME['risk_colors']['High'],
    annotation_text='Alert Threshold (+20%)'
)

fig.update_traces(
    line_width=PROJECT_THEME['line_width'],
    marker_size=PROJECT_THEME['marker_size'],
    line_color=PROJECT_THEME['highlight_blue']
)

fig = apply_project_theme(fig)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Performance Comparison: Anomaly Rate vs Actual Missing Rate
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Anomaly rate (bars)
fig.add_trace(
    go.Bar(
        x=monthly_metrics['month'],
        y=monthly_metrics['anomaly_rate'],
        name='Anomaly Rate (%)',
        marker_color=PROJECT_THEME['highlight_blue'],
        opacity=0.7
    ),
    secondary_y=False
)

# Actual missing rate (line)
fig.add_trace(
    go.Scatter(
        x=monthly_metrics['month'],
        y=monthly_metrics['avg_missing_rate'],
        name='Actual Missing Rate (%)',
        mode='lines+markers',
        line=dict(color=PROJECT_THEME['fraud_red'], width=PROJECT_THEME['line_width']),
        marker=dict(size=PROJECT_THEME['marker_size'])
    ),
    secondary_y=True
)

fig.update_layout(
    title='Model Predictions vs Actual Missing Rates',
    height=400
)
fig.update_yaxes(title_text='Anomaly Rate (%)', secondary_y=False)
fig.update_yaxes(title_text='Actual Missing Rate (%)', secondary_y=True)

fig = apply_project_theme(fig)
fig.show()

In [ ]:
# Calculate correlation between predicted anomalies and actual missing rate
correlation = monthly_metrics['anomaly_rate'].corr(monthly_metrics['avg_missing_rate'])
print(f"Correlation between anomaly rate and actual missing rate: {correlation:.3f}")

if correlation > 0.7:
    print("Model predictions are well aligned with actual fraud patterns.")
elif correlation > 0.4:
    print("Model shows moderate alignment. Consider monitoring for drift.")
else:
    print("WARNING: Low correlation suggests model may need retraining.")

## 4. Data Drift Detection

Monitor feature distributions for changes that could affect model performance.

In [ ]:
# Split data into reference (training) and current periods
# Using first 6 months as reference, last 6 months as current
months_list = sorted(ml_data['order_month'].unique())
mid_point = len(months_list) // 2

reference_months = months_list[:mid_point]
current_months = months_list[mid_point:]

reference_data = ml_data[ml_data['order_month'].isin(reference_months)]
current_data = ml_data[ml_data['order_month'].isin(current_months)]

print(f"Reference period: {reference_months[0]} to {reference_months[-1]} ({len(reference_data):,} records)")
print(f"Current period: {current_months[0]} to {current_months[-1]} ({len(current_data):,} records)")

In [ ]:
# Calculate drift metrics using Kolmogorov-Smirnov test
drift_results = []

for feature in available_features:
    ref_values = reference_data[feature].dropna()
    curr_values = current_data[feature].dropna()
    
    # KS test for distribution difference
    ks_stat, p_value = stats.ks_2samp(ref_values, curr_values)
    
    # Calculate mean shift
    mean_shift = ((curr_values.mean() - ref_values.mean()) / ref_values.mean()) * 100 if ref_values.mean() != 0 else 0
    
    # Determine drift status
    if p_value < 0.01:
        drift_status = 'Critical'
    elif p_value < 0.05:
        drift_status = 'High'
    elif p_value < 0.1:
        drift_status = 'Medium'
    else:
        drift_status = 'Low'
    
    drift_results.append({
        'feature': feature,
        'ks_statistic': ks_stat,
        'p_value': p_value,
        'mean_shift_pct': mean_shift,
        'ref_mean': ref_values.mean(),
        'curr_mean': curr_values.mean(),
        'drift_status': drift_status
    })

drift_df = pd.DataFrame(drift_results)
drift_df = drift_df.sort_values('ks_statistic', ascending=False)
print("Feature Drift Analysis:")
drift_df

In [ ]:
# Drift Status Distribution (Horizontal Bar Chart)
drift_counts = drift_df['drift_status'].value_counts().reset_index()
drift_counts.columns = ['drift_status', 'count']

# Order by risk level (descending for horizontal bars)
order = ['Critical', 'High', 'Medium', 'Low']
drift_counts['drift_status'] = pd.Categorical(drift_counts['drift_status'], categories=order, ordered=True)
drift_counts = drift_counts.sort_values('drift_status', ascending=False)

fig = px.bar(
    drift_counts,
    x='count',
    y='drift_status',
    orientation='h',
    title='Feature Drift Status Distribution',
    labels={'count': 'Number of Features', 'drift_status': 'Drift Status'},
    color='drift_status',
    color_discrete_map=PROJECT_THEME['risk_colors'],
    text='count'
)

fig.update_traces(textposition='outside')
fig.update_layout(height=350, showlegend=False)
fig = apply_project_theme(fig)
fig.show()

In [ ]:
# KS Statistic by Feature (Horizontal Bar Chart with Highlight)
drift_sorted = drift_df.sort_values('ks_statistic', ascending=True)
colors = get_top_n_highlight_colors(drift_sorted, top_n=3, ascending=False)

fig = px.bar(
    drift_sorted,
    x='ks_statistic',
    y='feature',
    orientation='h',
    title='Feature Drift: KS Statistic by Feature',
    labels={'ks_statistic': 'KS Statistic (0-1)', 'feature': 'Feature'},
    color=colors,
    color_discrete_map='identity',
    text='ks_statistic'
)

fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(height=450, showlegend=False)

# Add threshold line
fig.add_vline(
    x=0.1,
    line_dash='dash',
    line_color=PROJECT_THEME['risk_colors']['High'],
    annotation_text='Alert Threshold'
)

fig = apply_project_theme(fig)
fig.show()

In [ ]:
# Mean Shift Analysis (Horizontal Bar Chart)
shift_sorted = drift_df.sort_values('mean_shift_pct', key=abs, ascending=True)
shift_colors = [PROJECT_THEME['fraud_red'] if x > 0 else PROJECT_THEME['safe_green'] for x in shift_sorted['mean_shift_pct']]

fig = px.bar(
    shift_sorted,
    x='mean_shift_pct',
    y='feature',
    orientation='h',
    title='Feature Mean Shift: Reference vs Current Period',
    labels={'mean_shift_pct': 'Mean Shift (%)', 'feature': 'Feature'},
    color=shift_colors,
    color_discrete_map='identity',
    text='mean_shift_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=450, showlegend=False)

# Add zero line
fig.add_vline(x=0, line_color='black', line_width=1)

fig = apply_project_theme(fig)
fig.show()

In [ ]:
# Distribution comparison for top drifting feature
top_drift_feature = drift_df.iloc[0]['feature']

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=reference_data[top_drift_feature],
    name='Reference Period',
    opacity=0.7,
    marker_color=PROJECT_THEME['highlight_blue'],
    histnorm='probability'
))

fig.add_trace(go.Histogram(
    x=current_data[top_drift_feature],
    name='Current Period',
    opacity=0.7,
    marker_color=PROJECT_THEME['highlight_orange'],
    histnorm='probability'
))

fig.update_layout(
    title=f'Distribution Comparison: {top_drift_feature}',
    xaxis_title=f'{top_drift_feature}',
    yaxis_title='Probability',
    barmode='overlay',
    height=400
)

fig = apply_project_theme(fig)
fig.show()

## 5. Retraining Pipeline

Automated retraining triggers and model validation.

In [ ]:
# Define retraining triggers
RETRAINING_TRIGGERS = {
    'anomaly_rate_change': 0.20,  # 20% change in anomaly rate
    'feature_drift_threshold': 0.15,  # KS statistic > 0.15
    'correlation_drop': 0.20,  # 20% drop in prediction-actual correlation
    'max_days_since_training': 90  # 90 days max without retraining
}

def evaluate_retraining_need(drift_df, monthly_metrics, correlation):
    """
    Evaluate if model needs retraining based on multiple criteria.
    """
    triggers_fired = []
    
    # Check anomaly rate stability
    rate_change = abs(monthly_metrics['anomaly_rate'].iloc[-1] - monthly_metrics['anomaly_rate'].mean()) / monthly_metrics['anomaly_rate'].mean()
    if rate_change > RETRAINING_TRIGGERS['anomaly_rate_change']:
        triggers_fired.append(f"Anomaly rate changed by {rate_change*100:.1f}%")
    
    # Check feature drift
    critical_drift = drift_df[drift_df['ks_statistic'] > RETRAINING_TRIGGERS['feature_drift_threshold']]
    if len(critical_drift) > 0:
        triggers_fired.append(f"{len(critical_drift)} features with significant drift")
    
    # Check correlation
    if correlation < 0.5:  # Below 50% correlation
        triggers_fired.append(f"Low prediction-actual correlation: {correlation:.2f}")
    
    return triggers_fired

triggers = evaluate_retraining_need(drift_df, monthly_metrics, correlation)

if triggers:
    print("RETRAINING RECOMMENDED")
    print("Triggers fired:")
    for t in triggers:
        print(f"  - {t}")
else:
    print("Model performance is stable. No retraining needed.")

In [ ]:
# Simulate retraining with different periods
def train_model_on_period(data, feature_columns):
    """
    Train Isolation Forest model on given data.
    Returns model and key metrics.
    """
    X = data[feature_columns].fillna(data[feature_columns].median())
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    model = IsolationForest(
        n_estimators=100,
        contamination=0.1,
        random_state=42
    )
    model.fit(X_scaled)
    
    predictions = model.predict(X_scaled)
    anomaly_rate = (predictions == -1).mean() * 100
    
    return {
        'model': model,
        'scaler': scaler,
        'anomaly_rate': anomaly_rate,
        'n_samples': len(data)
    }

# Train on different time windows
training_results = []

for i, month in enumerate(months_list):
    # Expanding window: use all data up to current month
    train_data = ml_data[ml_data['order_month'] <= month]
    if len(train_data) > 100:  # Minimum samples required
        result = train_model_on_period(train_data, available_features)
        training_results.append({
            'training_month': str(month),
            'anomaly_rate': result['anomaly_rate'],
            'n_samples': result['n_samples']
        })

training_df = pd.DataFrame(training_results)
print("Retraining simulation results:")
training_df

In [ ]:
# Model Stability Over Retraining Windows
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Sample size (bars)
fig.add_trace(
    go.Bar(
        x=training_df['training_month'],
        y=training_df['n_samples'],
        name='Training Samples',
        marker_color=PROJECT_THEME['neutral_gray'],
        opacity=0.5
    ),
    secondary_y=False
)

# Anomaly rate (line)
fig.add_trace(
    go.Scatter(
        x=training_df['training_month'],
        y=training_df['anomaly_rate'],
        name='Anomaly Rate (%)',
        mode='lines+markers',
        line=dict(color=PROJECT_THEME['highlight_blue'], width=PROJECT_THEME['line_width']),
        marker=dict(size=PROJECT_THEME['marker_size'])
    ),
    secondary_y=True
)

fig.update_layout(
    title='Model Stability: Expanding Training Window Analysis',
    height=400
)
fig.update_yaxes(title_text='Training Samples (count)', secondary_y=False)
fig.update_yaxes(title_text='Anomaly Rate (%)', secondary_y=True)

fig = apply_project_theme(fig)
fig.show()

## 6. A/B Testing Framework

Compare champion and challenger models.

In [ ]:
# Train champion (baseline) and challenger (retrained) models
# Champion: trained on first half of data
# Challenger: trained on all data

champion_result = train_model_on_period(reference_data, available_features)
challenger_result = train_model_on_period(ml_data, available_features)

print("Champion Model (trained on reference period):")
print(f"  Training samples: {champion_result['n_samples']:,}")
print(f"  Anomaly rate: {champion_result['anomaly_rate']:.2f}%")
print()
print("Challenger Model (trained on all data):")
print(f"  Training samples: {challenger_result['n_samples']:,}")
print(f"  Anomaly rate: {challenger_result['anomaly_rate']:.2f}%")

In [ ]:
# Evaluate both models on current period
X_current = current_data[available_features].fillna(current_data[available_features].median())

# Champion predictions
X_champion_scaled = champion_result['scaler'].transform(X_current)
champion_preds = champion_result['model'].predict(X_champion_scaled)
champion_scores = champion_result['model'].decision_function(X_champion_scaled)

# Challenger predictions
X_challenger_scaled = challenger_result['scaler'].transform(X_current)
challenger_preds = challenger_result['model'].predict(X_challenger_scaled)
challenger_scores = challenger_result['model'].decision_function(X_challenger_scaled)

# Compare
ab_comparison = pd.DataFrame({
    'model': ['Champion', 'Challenger'],
    'anomaly_rate': [
        (champion_preds == -1).mean() * 100,
        (challenger_preds == -1).mean() * 100
    ],
    'avg_score': [champion_scores.mean(), challenger_scores.mean()],
    'score_std': [champion_scores.std(), challenger_scores.std()]
})

print("A/B Test Results on Current Period:")
ab_comparison

In [ ]:
# A/B Test Comparison (Horizontal Bar Chart)
fig = px.bar(
    ab_comparison,
    x='anomaly_rate',
    y='model',
    orientation='h',
    title='A/B Test: Champion vs Challenger Anomaly Rates',
    labels={'anomaly_rate': 'Anomaly Rate (%)', 'model': 'Model'},
    color='model',
    color_discrete_map={
        'Champion': PROJECT_THEME['highlight_blue'],
        'Challenger': PROJECT_THEME['highlight_orange']
    },
    text='anomaly_rate'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=300, showlegend=False)
fig = apply_project_theme(fig)
fig.show()

In [ ]:
# Score Distribution Comparison
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=champion_scores,
    name='Champion',
    opacity=0.7,
    marker_color=PROJECT_THEME['highlight_blue'],
    histnorm='probability'
))

fig.add_trace(go.Histogram(
    x=challenger_scores,
    name='Challenger',
    opacity=0.7,
    marker_color=PROJECT_THEME['highlight_orange'],
    histnorm='probability'
))

fig.update_layout(
    title='A/B Test: Score Distribution Comparison',
    xaxis_title='Anomaly Score',
    yaxis_title='Probability',
    barmode='overlay',
    height=400
)

fig = apply_project_theme(fig)
fig.show()

In [ ]:
# Statistical significance test
ks_stat, p_value = stats.ks_2samp(champion_scores, challenger_scores)

print(f"Statistical Significance Test (KS Test):")
print(f"  KS Statistic: {ks_stat:.4f}")
print(f"  P-value: {p_value:.4f}")
print()

if p_value < 0.05:
    print("The difference between models is STATISTICALLY SIGNIFICANT.")
    print("Recommendation: Evaluate challenger model for production deployment.")
else:
    print("No statistically significant difference between models.")
    print("Recommendation: Keep champion model in production.")

## 7. Monitoring Dashboard Metrics

In [ ]:
# Create summary metrics for monitoring dashboard
monitoring_summary = {
    'last_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_version': 'v1.0',
    'training_samples': len(ml_data),
    'current_anomaly_rate': anomaly_rate,
    'prediction_correlation': correlation,
    'features_with_drift': len(drift_df[drift_df['drift_status'].isin(['Critical', 'High'])]),
    'retraining_triggers_fired': len(triggers),
    'retraining_recommended': len(triggers) > 0,
    'champion_vs_challenger_diff': abs(
        ab_comparison[ab_comparison['model'] == 'Champion']['anomaly_rate'].values[0] -
        ab_comparison[ab_comparison['model'] == 'Challenger']['anomaly_rate'].values[0]
    )
}

print("Model Monitoring Summary")
print("=" * 40)
for key, value in monitoring_summary.items():
    print(f"{key}: {value}")

In [ ]:
# Model Health Indicators (Horizontal Bar Chart)
health_metrics = [
    {'metric': 'Prediction Correlation', 'value': correlation * 100, 'threshold': 70},
    {'metric': 'Feature Stability', 'value': (1 - len(drift_df[drift_df['drift_status'].isin(['Critical', 'High'])]) / len(drift_df)) * 100, 'threshold': 80},
    {'metric': 'Rate Stability', 'value': 100 - (training_df['anomaly_rate'].std() / training_df['anomaly_rate'].mean() * 100), 'threshold': 85},
]

health_df = pd.DataFrame(health_metrics)
# Map status: higher score = better health = Low risk (green)
health_df['status'] = health_df.apply(
    lambda x: 'Low' if x['value'] >= x['threshold'] else ('Medium' if x['value'] >= x['threshold'] * 0.8 else 'High'),
    axis=1
)

# Sort by value descending to show best metrics at top
health_df = health_df.sort_values('value', ascending=True)

fig = px.bar(
    health_df,
    x='value',
    y='metric',
    orientation='h',
    title='Model Health Indicators',
    labels={'value': 'Health Score (%)', 'metric': 'Metric'},
    color='status',
    color_discrete_map=PROJECT_THEME['risk_colors'],
    text='value'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=300, xaxis_range=[0, 110])
fig = apply_project_theme(fig)
fig.show()

## 8. Summary & Recommendations

In [ ]:
print("=" * 60)
print("MODEL MONITORING REPORT")
print("=" * 60)
print()
print("1. PERFORMANCE MONITORING")
print(f"   - Current anomaly rate: {anomaly_rate:.2f}%")
print(f"   - Prediction-actual correlation: {correlation:.3f}")
print(f"   - Monthly rate variation: {monthly_metrics['anomaly_rate'].std():.2f}%")
print()
print("2. DATA DRIFT ANALYSIS")
print(f"   - Features analyzed: {len(available_features)}")
print(f"   - Critical drift: {len(drift_df[drift_df['drift_status'] == 'Critical'])}")
print(f"   - High drift: {len(drift_df[drift_df['drift_status'] == 'High'])}")
print(f"   - Top drifting feature: {drift_df.iloc[0]['feature']}")
print()
print("3. RETRAINING ASSESSMENT")
if triggers:
    print("   STATUS: RETRAINING RECOMMENDED")
    for t in triggers:
        print(f"   - {t}")
else:
    print("   STATUS: Model performing within acceptable parameters")
print()
print("4. A/B TEST RESULTS")
print(f"   - Champion anomaly rate: {ab_comparison[ab_comparison['model']=='Champion']['anomaly_rate'].values[0]:.2f}%")
print(f"   - Challenger anomaly rate: {ab_comparison[ab_comparison['model']=='Challenger']['anomaly_rate'].values[0]:.2f}%")
print(f"   - Statistical significance: {'Yes' if p_value < 0.05 else 'No'} (p={p_value:.4f})")
print()
print("=" * 60)

---

**Next Steps:**
1. Schedule monthly drift monitoring runs
2. Set up automated alerts for trigger conditions
3. Implement MLflow model registry for version control
4. Create production deployment pipeline with gradual rollout

**Dependencies for Production:**
- MLflow server for model versioning
- Airflow/Prefect for scheduling
- Alerting system (Slack, PagerDuty, etc.)